In [30]:
# Connexion directe à PostgreSQL avec psycopg2, requêtes SQL brutes
# Objectif : démontrer la connexion, quelques SELECT/UPDATE/INSERT/DELETE
# en respectant les bonnes pratiques de sécurité (variables d'environnement)

import os
import psycopg2
from dotenv import load_dotenv
import pandas as pd

In [31]:
# Chargement du fichier .env à partir de la racine du projet
# (le notebook se trouve dans /notebooks, le .env dans la racine)

env_path = os.path.abspath(os.path.join(os.getcwd(), "..", ".env"))
print("Chemin du .env :", env_path)

load_dotenv(env_path)

print("POSTGRES_DB :", os.getenv("POSTGRES_DB"))
print("POSTGRES_USER :", os.getenv("POSTGRES_USER"))
print("POSTGRES_PASSWORD est définie :", os.getenv("POSTGRES_PASSWORD") is not None)

Chemin du .env : /Users/natyferreira/SQL-mobigreen/.env
POSTGRES_DB : mobigreen_urban
POSTGRES_USER : admin
POSTGRES_PASSWORD est définie : True


In [32]:
# Connexion simple à PostgreSQL (sans exposer les identifiants)
# Toutes les informations sensibles viennent du .env

try:
    conn = psycopg2.connect(
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD"),
        host="localhost",
        port="5432"
    )
    cursor = conn.cursor()
    print("Connexion établie avec succès.")
except Exception as e:
    print("Erreur de connexion :", e)

Connexion établie avec succès.


In [33]:
# SELECT simple sur la table usagers
# Traz os usuários com ID, sobrenome, nome e email - começando pelo sobrenome

cursor.execute("""
SELECT usr_id, nom, prenom, email
FROM usagers
ORDER BY nom;
""")
rows = cursor.fetchall()

rows[:10]  # aperçu des 10 premiers enregistrements

[(18, 'Bernard', 'Emma', 'user18@example.com'),
 (5, 'Bernard', 'Hugo', 'user5@example.com'),
 (1, 'Bernard', 'Lucas', 'user1@example.com'),
 (3, 'Bernard', 'Louis', 'user3@example.com'),
 (19, 'Bernard', 'Alice', 'user19@example.com'),
 (6, 'Dubois', 'Alice', 'user6@example.com'),
 (8, 'Dubois', 'Lucas', 'user8@example.com'),
 (9, 'Dubois', 'Alice', 'user9@example.com'),
 (11, 'Dubois', 'Léa', 'user11@example.com'),
 (20, 'Dubois', 'Hugo', 'user20@example.com')]

In [34]:
# Affichage des résultats sous forme de DataFrame pour une meilleure lisibilité

df_usagers = pd.DataFrame(rows, columns=["usr_id", "nom", "prenom", "email"])
df_usagers.head(10)

,usr_id,nom,prenom,email
0,18,Bernard,Emma,user18@example.com
1,5,Bernard,Hugo,user5@example.com
2,1,Bernard,Lucas,user1@example.com
3,3,Bernard,Louis,user3@example.com
4,19,Bernard,Alice,user19@example.com
5,6,Dubois,Alice,user6@example.com
6,8,Dubois,Lucas,user8@example.com
7,9,Dubois,Alice,user9@example.com
8,11,Dubois,Léa,user11@example.com
9,20,Dubois,Hugo,user20@example.com


In [35]:
# Exemple d'UPDATE avec gestion d'erreurs et rollback automatique
# (exemple purement démonstratif, à adapter selon vos données réelles)

try:
    cursor.execute("""
        UPDATE usagers
        SET email = email
        WHERE usr_id = 1;
    """)
    conn.commit()
    print("UPDATE effectué avec succès.")
except Exception as e:
    print("Erreur pendant l'UPDATE :", e)
    conn.rollback()
    print("Rollback exécuté.")

UPDATE effectué avec succès.


In [36]:
# Exemple d'INSERT avec gestion d'erreurs et rollback
# (exemple : insertion d'un usager fictif)

try:
    cursor.execute("""
        INSERT INTO usagers (usr_id, nom, prenom, email)
        VALUES (9999, 'TestNom', 'TestPrenom', 'test@example.com');
    """)
    conn.commit()
    print("INSERT effectué avec succès.")
except Exception as e:
    print("Erreur pendant l'INSERT :", e)
    conn.rollback()
    print("Rollback exécuté.")

Erreur pendant l'INSERT : null value in column "type_abonnement" of relation "usagers" violates not-null constraint
DETAIL:  Failing row contains (9999, TestNom, TestPrenom, test@example.com, null, null, null, null, null).

Rollback exécuté.


In [37]:
# Exemple de DELETE avec gestion d'erreurs et rollback
# (suppression de l'usager fictif inséré ci-dessus)

try:
    cursor.execute("""
        DELETE FROM usagers
        WHERE usr_id = 9999;
    """)
    conn.commit()
    print("DELETE effectué avec succès.")
except Exception as e:
    print("Erreur pendant le DELETE :", e)
    conn.rollback()
    print("Rollback exécuté.")

DELETE effectué avec succès.


In [38]:
# Fermeture propre de la connexion "manuelle"
# (avant de passer à une version plus robuste avec context manager)

try:
    cursor.close()
    conn.close()
    print("Connexion fermée proprement.")
except Exception as e:
    print("Erreur lors de la fermeture de la connexion :", e)

Connexion fermée proprement.


In [39]:
# Version robuste : Context Manager pour la connexion à PostgreSQL
# - Ouverture et fermeture automatiques
# - Commit si tout va bien
# - Rollback automatique en cas d'erreur
# - Aucune fuite de connexion

import time

class DatabaseConnection:
    def __enter__(self):
        self.conn = psycopg2.connect(
            dbname=os.getenv("POSTGRES_DB"),
            user=os.getenv("POSTGRES_USER"),
            password=os.getenv("POSTGRES_PASSWORD"),
            host="localhost",
            port="5432"
        )
        self.cursor = self.conn.cursor()
        print("Connexion ouverte avec succès.")
        return self

    def execute(self, query, params=None):
        start = time.time()
        try:
            self.cursor.execute(query, params)
            duration = time.time() - start
            print(f"Requête exécutée en {duration:.4f} secondes.")
        except Exception as e:
            print("Erreur pendant l'exécution de la requête :", e)
            self.conn.rollback()
            print("Rollback automatique exécuté.")
            raise

    def fetchall(self):
        return self.cursor.fetchall()

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            self.conn.commit()
            print("Commit réalisé.")
        else:
            self.conn.rollback()
            print("Erreur détectée. Rollback réalisé.")
            print(f"Détails de l'erreur : {exc_value}")

        self.cursor.close()
        self.conn.close()
        print("Connexion fermée.")

In [40]:
# Utilisation du context manager pour un SELECT

with DatabaseConnection() as db:
    db.execute("""
        SELECT usr_id, nom, prenom, email
        FROM usagers
        ORDER BY nom;
    """)
    rows_cm = db.fetchall()

df_usagers_cm = pd.DataFrame(rows_cm, columns=["usr_id", "nom", "prenom", "email"])
df_usagers_cm.head(10)

Connexion ouverte avec succès.
Requête exécutée en 0.0019 secondes.
Commit réalisé.
Connexion fermée.


,usr_id,nom,prenom,email
0,18,Bernard,Emma,user18@example.com
1,5,Bernard,Hugo,user5@example.com
2,1,Bernard,Lucas,user1@example.com
3,3,Bernard,Louis,user3@example.com
4,19,Bernard,Alice,user19@example.com
5,6,Dubois,Alice,user6@example.com
6,8,Dubois,Lucas,user8@example.com
7,9,Dubois,Alice,user9@example.com
8,11,Dubois,Léa,user11@example.com
9,20,Dubois,Hugo,user20@example.com


In [41]:
# Utilisation du context manager pour un UPDATE (exemple démonstratif)

with DatabaseConnection() as db:
    db.execute("""
        UPDATE usagers
        SET email = email
        WHERE usr_id = 1;
    """)
    # Pas besoin d'appeler commit() : il est géré dans __exit__

Connexion ouverte avec succès.
Requête exécutée en 0.0020 secondes.
Commit réalisé.
Connexion fermée.


In [42]:
# Utilisation du context manager pour un INSERT + DELETE de test
# (version corrigée avec toutes les colonnes NOT NULL)

with DatabaseConnection() as db:
    db.execute("""
        INSERT INTO usagers (
            usr_id, nom, prenom, email, telephone,
            type_abonnement, date_inscription,
            email_hash, telephone_chiffre
        )
        VALUES (
            9998, 'TempNom', 'TempPrenom', 'temp@example.com',
            '0000000000', 'standard', NOW(),
            'placeholder', 'placeholder'
        );
    """)

with DatabaseConnection() as db:
    db.execute("""
        DELETE FROM usagers
        WHERE usr_id = 9998;
    """)

Connexion ouverte avec succès.
Requête exécutée en 0.0017 secondes.
Commit réalisé.
Connexion fermée.
Connexion ouverte avec succès.
Requête exécutée en 0.0022 secondes.
Commit réalisé.
Connexion fermée.


In [43]:
# Fin du Notebook 1
# - Connexion sécurisée via .env
# - Aucune information sensible dans le code
# - Requêtes de base (SELECT/UPDATE/INSERT/DELETE)
# - Context manager robuste avec rollback automatique
#
# Prêt pour un commit sécurisé sur GitHub.